# BODAQS Widget Test Notebook

This notebook consumes artifacts produced by the batch pre-processing pipeline and displays the data using generic widgets.

 - Metric histogram widget
 - Event browser widget
 - Metric scatter widget
 - Signal histogram widget


In [6]:
from IPython.display import display
from bodaqs_analysis.widgets.session_selector import make_session_selector
from bodaqs_analysis.schema import load_event_schema
from bodaqs_analysis.artifacts import load_session_artifacts

def session_loader(session_key: str) -> dict:
    try:
        run_id, session_id = key_to_ref[session_key]
    except KeyError as e:
        raise KeyError(
            f"Unknown session_key {session_key!r}. "
            "Did the session selection change?"
        ) from e

    return load_session_artifacts(
        store,
        run_id=run_id,
        session_id=session_id,
    )
    
SCHEMA_PATH = r"event schema\event_schema.yaml"
schema = load_event_schema(SCHEMA_PATH)

sel = make_session_selector(artifacts_dir="artifacts", select_first_by_default=True)
display(sel["ui"])

events_index_df = sel["get_events_index_df"]()
key_to_ref = sel["get_key_to_ref"]()
store = sel["store"]


In [7]:
from bodaqs_analysis.widgets.signal_histogram_widget import make_signal_histogram_widget_for_loader

wh = make_signal_histogram_widget_for_loader(
    events_index_df,
    session_loader=session_loader,
    session_key_col="session_key",
    default_bins=50,
)

In [8]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader
from bodaqs_analysis.artifacts import load_all_events_for_selected

# ---- Load events for selected sessions ----

if not key_to_ref:
    raise ValueError("No sessions selected. Select sessions first.")

events_df_sel = load_all_events_for_selected(store, key_to_ref=key_to_ref)

print("Loaded events_df_sel:", events_df_sel.shape)

wb = make_event_browser_widget_for_loader(
    schema,
    events_df_sel,                 # concatenated events with session_key column
    session_loader=session_loader,
    session_key_col="session_key", # <-- NEW
)

Loaded events_df_sel: (28, 38)


In [ ]:
from bodaqs_analysis.widgets.metric_scatter_widget import make_metric_scatter_widget

handles = make_metric_scatter_widget(
    events_df=events_df,
    metrics_df=metrics_df,
    event_type_col="event_name",   # or "event_type"/"schema_id" depending on your table
    signal_col="signal_col",
)

viz_df = handles["viz_df"]  # handy for debugging
